# LEBIS — การค้นหาและการนับ (Search & Count)
### งานส่งวิชา Data Structures & Algorithms — โมดูลของ Neo

**ผู้เขียน:** Intuch Srisawat (Neo) · รหัส 68011345 · KMITL Robotics & AI Engineering

โน้ตบุ๊กนี้คือส่วน **search + count** ของ LEBIS ซึ่งเป็นงานที่ถูกให้คะแนนของ Neo
อัลกอริทึมที่เขียนเองในไฟล์นี้คือ **binary search** (ห้ามใช้ `bisect` ของ Python)
ส่วน `linear search` และ hash lookup มีไว้เป็นตัวเทียบเวลา

#### สิ่งที่โน้ตบุ๊กนี้พิสูจน์
1. binary search / linear search / การนับ ทำงาน **ถูกต้อง** (มี `verify()` ในตัว)
2. เวลาที่วัดได้จริงบนชุด **A–D** สอดคล้องกับ Big-O ที่อ้าง
3. ที่ข้อมูลน้อย (10 รายการ) linear เร็วกว่าหรือพอ ๆ กับ binary — ที่ 1,000 รายการ binary ชนะขาด
4. เทียบ hand-written กับของที่ Python มีให้ (`bisect` · `dict`) เพื่อยืนยันว่าเขียนมือทำงานถูกและเข้าใจว่าทำไม built-in เร็วกว่า

> **กติกาไฟล์วิชานี้:** ทุกเซลล์ใช้ **standard library ของ Python เท่านั้น** ไม่ import FastAPI / database / ไฟล์อื่นในโปรเจกต์
> รันเซลล์ **ตามลำดับบนลงล่าง** — เซลล์ `datasets` ต้องมาก่อนเซลล์ `search` เสมอ

---
## 1. ชุดข้อมูลทดสอบ A–D (`datasets.py`)

ทั้งสามหัวข้อของกลุ่ม (hash_table ของ Kim · search ของ Neo · sort_group ของ Kul) วัดผลบน
`make_dataset()` **ตัวเดียวกัน** ตัวเลขจึงอ่านเทียบกันได้ วันอ้างอิงถูกตรึงไว้ที่ `TODAY = 2026-10-01`
ตัวเลขในรายงานกับในวิดีโอจึงตรงกันทุกครั้งที่รัน

| ชุด | ข้อมูล | ใช้ทดสอบ |
|---|---|---|
| **A** | 10 รายการ สะอาด | ผลลัพธ์ปกติ |
| **B** | 12 รายการ ชื่อซ้ำเป็นคู่ (code ไม่ซ้ำ) | การค้นเมื่อชื่อซ้ำ |
| **C** | 10 รายการ ของหมดสต็อกครึ่งหนึ่ง | ค่าขอบ (ศูนย์) |
| **D** | 1,000 รายการ | วัดเวลาจริงเทียบ Big-O |

In [1]:
"""ชุดข้อมูลทดสอบ A–D — งานส่งวิชา Data Structures & Algorithms ของ Neo

ทั้งสามหัวข้อของกลุ่ม (hash_table ของ Kim · search ของ Neo · sort_group ของ Kul)
ต้องวัดผลบนข้อมูลชุดเดียวกันจากไฟล์นี้ ไม่ใช่ต่างคนต่างสร้างข้อมูลของตัวเอง
ถ้าแต่ละคนวัดบนข้อมูลคนละชุด ตัวเลขในรายงานจะอ่านเทียบกันไม่ได้เลย
และกรรมการจะเห็นทันทีว่าสามหัวข้อไม่ได้คุยกัน

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

วิธีรัน
    python3 backend-api/algorithms/datasets.py

วิธีใช้ใน Colab
    วางเซลล์ของไฟล์นี้ไว้ "ก่อน" เซลล์ของ hash_table.py และ search.py
    แล้วเรียก make_dataset("A") ... make_dataset("D") ส่งเข้าฟังก์ชันของแต่ละคน
    ได้ตรงๆ เพราะทุกตัวรับ list ของ dict ธรรมดา

หนึ่ง record มี 8 ฟิลด์ตามที่ Kim ออกแบบไว้
    code       รหัสอุปกรณ์ ไม่ซ้ำกันทั้งชุด — ใช้เป็น key ของตารางแฮช
    name       ชื่อที่คนเรียก ซ้ำกันได้ (ชุด B จงใจให้ซ้ำ)
    category   หมวด ใช้ตอนจัดกลุ่มของ Kul
    total      จำนวนทั้งหมดที่ร้านมี
    available  จำนวนที่ยังอยู่บนชั้นตอนนี้
    borrower   ชื่อผู้เช่า ถ้ายังไม่มีใครยืมเป็น None
    due_date   กำหนดคืนในรูป "YYYY-MM-DD" ถ้าไม่มีคนยืมเป็น None
    status     "ok" ว่างให้เช่าได้ · "out" ถูกยืมหมด · "broken" ชำรุด

ทำไมวันที่ถึงตายตัว
    count_overdue() ของ search.py นับจากวันครบกำหนดเทียบกับ "วันนี้"
    ถ้าใช้วันจริงของเครื่อง ตัวเลขในรายงานจะเปลี่ยนทุกวันที่รันใหม่
    และตัวเลขในรายงานกับตัวเลขในวิดีโอจะไม่ตรงกัน ไฟล์นี้จึงตรึง TODAY ไว้
    ให้ทุกคนอ้างอิงค่าเดียวกัน
"""

import random
from datetime import date, timedelta

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — ค่าคงที่
# ---------------------------------------------------------------------------

# วันอ้างอิงตายตัว ใช้แทน "วันนี้" ทั้งระบบทดสอบ
# ตัวเลขในรายงานจึงเท่าเดิมทุกครั้งที่รัน ไม่ว่าจะรันวันไหน
TODAY = date(2026, 10, 1)

# หมวดและคำนำหน้ารหัส ใช้ชุดเดียวกับ hash_table.py ของ Kim
# (คัดลอกค่ามา ไม่ได้ import เพราะกฎ stdlib เท่านั้นข้างบน)
CATEGORIES = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]

CATEGORY_PREFIX = {
    "กล้อง": "CAM",
    "เลนส์": "LEN",
    "ขาตั้ง": "TRI",
    "ไฟ": "LED",
    "ไมโครโฟน": "MIC",
}

# ชื่อผู้เช่าสำหรับข้อมูลจำลอง ไม่ใช่ชื่อคนจริง
BORROWERS = [
    "ปวีณา",
    "ธนกร",
    "สุชาดา",
    "ณัฐพล",
    "กมลชนก",
    "อานนท์",
    "พิมพ์ชนก",
    "วีรภัทร",
]

# seed แยกตามชุด ชุดเดิมจึงได้ข้อมูลเดิมเสมอ แต่คนละชุดไม่เหมือนกัน
SEEDS = {"A": 11, "B": 22, "C": 33, "D": 44}

DATASET_SIZES = {"A": 10, "B": 12, "C": 10, "D": 1000}


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — ตัวช่วยสร้าง record
# ---------------------------------------------------------------------------


def _iso(day: date) -> str:
    """แปลงวันเป็นข้อความ "YYYY-MM-DD"

    เก็บเป็นข้อความไม่ใช่ออบเจ็กต์ date เพราะรูปแบบนี้เรียงลำดับด้วยการเทียบ
    ข้อความได้ตรงๆ อยู่แล้ว (ปี-เดือน-วัน เรียงจากหน่วยใหญ่ไปเล็ก)
    merge_sort ของ Kul กับ binary_search ของ Neo จึงเทียบวันได้โดยไม่ต้องแปลงก่อน
    """
    return day.isoformat()


def _make_record(
    number: int,
    category: str,
    name: str,
    total: int,
    available: int,
    borrower=None,
    due_offset=None,
    broken: bool = False,
) -> dict:
    """ประกอบ record 8 ฟิลด์ให้ครบ และคิดค่า status ให้สอดคล้องกับ available

    due_offset คือจำนวนวันนับจาก TODAY ติดลบ = เลยกำหนดคืนมาแล้ว
    """
    if broken:
        status = "broken"
    elif available == 0:
        status = "out"
    else:
        status = "ok"

    return {
        "code": f"{CATEGORY_PREFIX[category]}-{number:04d}",
        "name": name,
        "category": category,
        "total": total,
        "available": available,
        "borrower": borrower,
        "due_date": _iso(TODAY + timedelta(days=due_offset)) if due_offset is not None else None,
        "status": status,
    }


def _build(size: int, seed: int, duplicate_names: bool, zero_stock: bool) -> list:
    """สร้างรายการตามพารามิเตอร์ที่แต่ละชุดต้องการ

    duplicate_names  ให้ชื่อซ้ำกันเป็นคู่ๆ แต่ code ยังไม่ซ้ำ (ชุด B)
    zero_stock       ดันให้หลายตัวมี available = 0 (ชุด C)
    """
    generator = random.Random(seed)
    items = []

    for number in range(1, size + 1):
        # ชุด B: ตัวเลขคู่ใช้ชื่อเดียวกับตัวก่อนหน้า จึงได้ชื่อซ้ำเป็นคู่ๆ
        # แต่ code ยังไม่ซ้ำ เพราะ code เดินตาม number ตลอด
        # ใช้พิสูจน์ว่าค้นด้วยชื่อแล้วยังแยกออกว่าเป็นตัวไหน
        #
        # หมวดต้องเดินตามชื่อไปด้วย ไม่งั้นจะได้ของที่ชื่อว่า "เลนส์ รุ่นที่ 1"
        # แต่หมวดเป็น "ขาตั้ง" ซึ่งเป็นข้อมูลที่ขัดกันเองและทำให้ group_by_type
        # ของ Kul ให้ผลที่อธิบายในรายงานไม่ได้
        name_number = number - 1 if duplicate_names and number % 2 == 0 else number
        category = CATEGORIES[name_number % len(CATEGORIES)]
        name = f"{category} รุ่นที่ {name_number}"

        total = generator.randint(1, 5)

        if zero_stock and number % 2 == 1:
            # ชุด C: ครึ่งหนึ่งถูกยืมไปหมด
            available = 0
        else:
            available = generator.randint(0, total)

        borrowed_units = total - available

        if borrowed_units > 0:
            borrower = BORROWERS[number % len(BORROWERS)]
            # กระจายกำหนดคืนไว้ทั้งก่อนและหลังวันอ้างอิง
            # ค่าติดลบคือเลยกำหนดแล้ว ซึ่งเป็นตัวที่ count_overdue() ต้องนับได้
            due_offset = generator.randint(-10, 14)
        else:
            borrower = None
            due_offset = None

        # ให้มีของชำรุดปนอยู่บ้าง เพื่อให้ count_by_status() มีครบทั้งสามสถานะ
        # ทุกๆ 7 ชิ้น — เลือกเลขนี้เพราะชุดเล็กสุดมี 10 ชิ้น ใช้ 7 แล้วยังได้
        # อย่างน้อย 1 ชิ้นในทุกชุด ถ้าใช้เลขที่ใหญ่กว่านั้น ชุด A-C จะไม่มีของชำรุดเลย
        # แล้วการนับสถานะ "broken" จะไม่เคยถูกทดสอบบนชุดที่ตรวจด้วยตาได้
        broken = number % 7 == 0

        items.append(
            _make_record(
                number=number,
                category=category,
                name=name,
                total=total,
                available=available,
                borrower=borrower,
                due_offset=due_offset,
                broken=broken,
            )
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — ทางเข้าหลัก
# ---------------------------------------------------------------------------


def make_dataset(name: str) -> list:
    """คืนชุดข้อมูลทดสอบตามชื่อชุด "A" "B" "C" หรือ "D"

    A  10 รายการ  ข้อมูลสะอาด ชื่อไม่ซ้ำ       ใช้เช็กว่าทุกฟังก์ชันทำงานถูก
    B  12 รายการ  ชื่อซ้ำเป็นคู่ code ไม่ซ้ำ    ใช้เช็กว่าค้นแล้วได้ตัวที่ถูก
    C  10 รายการ  หลายตัว available = 0        ใช้เช็กการนับและสถานะ
    D  1,000 รายการ                            ใช้วัดเวลาจริงเทียบ Big-O

    เรียกซ้ำกี่ครั้งก็ได้ข้อมูลเดิมเสมอ เพราะ seed ตายตัวตาม SEEDS
    แต่คืน list ใหม่ทุกครั้ง ฟังก์ชันที่แก้ข้อมูล (เช่น sort แบบแก้ในที่)
    จึงไม่ไปกระทบผลของคนอื่นที่เรียกทีหลัง
    """
    key = name.strip().upper()

    if key not in DATASET_SIZES:
        raise ValueError(f"ไม่รู้จักชุดข้อมูล {name!r} — มีให้เลือกแค่ A, B, C, D")

    return _build(
        size=DATASET_SIZES[key],
        seed=SEEDS[key],
        duplicate_names=(key == "B"),
        zero_stock=(key == "C"),
    )


def all_datasets() -> dict:
    """คืนทั้งสี่ชุดในทีเดียว ใช้ตอนวนวัดผลในรายงาน"""
    return {name: make_dataset(name) for name in ("A", "B", "C", "D")}


def describe(items: list) -> dict:
    """สรุปหน้าตาของชุดข้อมูลไว้ตรวจว่าสร้างมาถูกจริง

    ไม่ได้เรียกฟังก์ชันนับของ search.py เพราะไฟล์นี้ต้องรันได้ลำพัง
    (และ search.py ก็ต้องรันได้ลำพังเหมือนกัน) การนับสองบรรทัดตรงนี้
    จึงถูกกว่าการผูกสองไฟล์เข้าหากัน
    """
    codes = [item["code"] for item in items]
    names = [item["name"] for item in items]

    return {
        "จำนวน": len(items),
        "code ไม่ซ้ำ": len(set(codes)) == len(codes),
        "ชื่อที่ซ้ำกัน": len(names) - len(set(names)),
        "available = 0": sum(1 for item in items if item["available"] == 0),
        "มีคนยืมอยู่": sum(1 for item in items if item["borrower"] is not None),
        "เลยกำหนดคืน": sum(
            1
            for item in items
            if item["due_date"] is not None and item["due_date"] < _iso(TODAY)
        ),
        "ชำรุด": sum(1 for item in items if item["status"] == "broken"),
    }


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — ตรวจว่าแต่ละชุดมีคุณสมบัติตามที่ตั้งใจไว้จริง
#
# ข้อมูลทดสอบที่ผิดเงียบๆ อันตรายกว่าไม่มีข้อมูลทดสอบ เพราะทุกฟังก์ชัน
# จะ "ผ่าน" หมดแล้วเราจะเชื่อผลนั้นไปเขียนในรายงาน
# ---------------------------------------------------------------------------


def verify() -> None:
    """เช็กคุณสมบัติที่แต่ละชุดสัญญาไว้ ถ้าไม่ตรงให้ล้มทันที"""
    a = make_dataset("A")
    b = make_dataset("B")
    c = make_dataset("C")
    d = make_dataset("D")

    # ขนาดตามที่แผนกำหนด — โจทย์บังคับว่าทุกชุดต้อง >= 10
    assert len(a) == 10 and len(b) == 12 and len(c) == 10 and len(d) == 1000
    assert all(len(items) >= 10 for items in (a, b, c, d))

    # code ต้องไม่ซ้ำทุกชุด เพราะเป็น key ของตารางแฮช
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        codes = [item["code"] for item in items]
        assert len(set(codes)) == len(codes), f"ชุด {name} มี code ซ้ำ"

    # ชุด A ชื่อต้องไม่ซ้ำ ชุด B ต้องซ้ำ — นี่คือความต่างที่ชุด B มีไว้พิสูจน์
    assert len({item["name"] for item in a}) == len(a), "ชุด A ไม่ควรมีชื่อซ้ำ"
    assert len({item["name"] for item in b}) < len(b), "ชุด B ต้องมีชื่อซ้ำ"

    # ชุด C ต้องมีของหมดสต็อกอย่างน้อยครึ่งหนึ่ง
    assert sum(1 for item in c if item["available"] == 0) >= len(c) // 2

    # ทุกชุดต้องมีของชำรุดอย่างน้อยหนึ่งชิ้น ไม่งั้น count_by_status()
    # จะไม่เคยถูกทดสอบครบทั้งสามสถานะ
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        assert any(item["status"] == "broken" for item in items), f"ชุด {name} ไม่มีของชำรุด"

    # ทุกชุดต้องมีของที่เลยกำหนดคืน ไม่งั้น count_overdue() จะทดสอบไม่ได้
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        overdue = [
            item
            for item in items
            if item["due_date"] is not None and item["due_date"] < _iso(TODAY)
        ]
        assert overdue, f"ชุด {name} ไม่มีรายการที่เลยกำหนดคืนเลย"

    # ความสอดคล้องของ record ทุกใบ
    for items in (a, b, c, d):
        for item in items:
            assert set(item) == {
                "code",
                "name",
                "category",
                "total",
                "available",
                "borrower",
                "due_date",
                "status",
            }, "record ต้องมีครบ 8 ฟิลด์"
            assert 0 <= item["available"] <= item["total"], "ของว่างต้องไม่เกินของทั้งหมด"
            assert item["status"] in ("ok", "out", "broken")

            # มีคนยืมก็ต้องมีกำหนดคืน ไม่มีคนยืมก็ต้องไม่มีกำหนดคืน
            assert (item["borrower"] is None) == (item["due_date"] is None)

            # ของที่ยังไม่มีใครยืมต้องมีของครบบนชั้น
            if item["borrower"] is None:
                assert item["available"] == item["total"]

    # เรียกสองครั้งต้องได้ข้อมูลเดิมเป๊ะ ไม่งั้นตัวเลขในรายงานจะขยับเอง
    assert make_dataset("D") == d, "make_dataset ต้องคืนข้อมูลเดิมทุกครั้ง"

    # แต่ต้องเป็น list คนละใบ ไม่ใช่ใบเดิมที่ใครแก้ไปแล้ว
    assert make_dataset("A") is not a

    print("ตรวจชุดข้อมูล A-D ผ่านทั้งหมด")


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — รันเพื่อดูผล
# ---------------------------------------------------------------------------


def main() -> None:
    print("=" * 70)
    print("ชุดข้อมูลทดสอบ A-D ของ LEBIS")
    print(f"วันอ้างอิง (TODAY) = {_iso(TODAY)}")
    print("=" * 70)

    for name in ("A", "B", "C", "D"):
        items = make_dataset(name)
        print()
        print(f"ชุด {name}")
        print("-" * 70)
        for label, value in describe(items).items():
            print(f"    {label:<16} {value}")

    print()
    print("ตัวอย่าง 3 รายการแรกของชุด A")
    print("-" * 70)
    for item in make_dataset("A")[:3]:
        print(f"    {item}")

    print()
    print("ชื่อที่ซ้ำกันในชุด B (code ไม่ซ้ำ)")
    print("-" * 70)
    for item in make_dataset("B"):
        print(f"    {item['code']}  {item['name']}")

    print()
    verify()


รันชุดข้อมูลและตรวจว่ามีคุณสมบัติตามที่ออกแบบไว้จริง (`verify()` ล้มทันทีถ้าไม่ตรง)

In [2]:
TODAY_STR = TODAY.isoformat()
print(f"วันอ้างอิง (TODAY) = {TODAY_STR}\n")

for name in ("A", "B", "C", "D"):
    items = make_dataset(name)
    print(f"ชุด {name}  (จำนวน {len(items)})")
    for label, value in describe(items).items():
        print(f"    {label:<16} {value}")
    print()

verify()   # ตรวจคุณสมบัติของทุกชุด

วันอ้างอิง (TODAY) = 2026-10-01

ชุด A  (จำนวน 10)
    จำนวน            10
    code ไม่ซ้ำ      True
    ชื่อที่ซ้ำกัน    0
    available = 0    1
    มีคนยืมอยู่      7
    เลยกำหนดคืน      3
    ชำรุด            1

ชุด B  (จำนวน 12)
    จำนวน            12
    code ไม่ซ้ำ      True
    ชื่อที่ซ้ำกัน    6
    available = 0    5
    มีคนยืมอยู่      8
    เลยกำหนดคืน      4
    ชำรุด            1

ชุด C  (จำนวน 10)
    จำนวน            10
    code ไม่ซ้ำ      True
    ชื่อที่ซ้ำกัน    0
    available = 0    5
    มีคนยืมอยู่      7
    เลยกำหนดคืน      4
    ชำรุด            1

ชุด D  (จำนวน 1000)
    จำนวน            1000
    code ไม่ซ้ำ      True
    ชื่อที่ซ้ำกัน    0
    available = 0    288
    มีคนยืมอยู่      685
    เลยกำหนดคืน      275
    ชำรุด            142

ตรวจชุดข้อมูล A-D ผ่านทั้งหมด


---
## 2. การค้นหาและการนับ (`search.py`)

### Big-O ของแต่ละฟังก์ชัน

| ฟังก์ชัน | เวลา | หน่วยความจำ | เขียนเอง? |
|---|---|---|---|
| `linear_search` | O(n) | O(1) | ✔ ตัวเทียบ |
| **`binary_search`** | **O(log n)** | **O(1)** | ✔ **ตัวที่ถูกให้คะแนน** (ลูป ไม่ recursion จึงไม่กิน stack) |
| `hash_lookup` | O(1) เฉลี่ย / O(n) แย่สุด | O(n) ที่ตาราง | ใช้ตารางของ Kim |
| `search_by_name` | O(n) | O(k) ที่ผลลัพธ์ | ✔ คืน **ทุก** ตัวที่ตรง |
| `find_due_before` | O(log n + k) | O(k) | ✔ คำถามแบบ **ช่วง** ที่แฮชตอบไม่ได้ |
| `count_available` / `count_by_status` / `count_overdue` | O(n) | O(1) | ✔ ต้องดูทุกชิ้น ไม่มีทางลัด |

**ทำไม `binary_search` เป็น O(log n):** ทุกครั้งที่เทียบ 1 รอบ จำนวนที่เหลือต้องดูจะลดลงครึ่งหนึ่ง
ข้อมูล 1,000 ชิ้นจึงเทียบแค่ ~10 ครั้ง (เพราะ 2¹⁰ = 1,024) เทียบกับ linear ที่เทียบเฉลี่ย ~500 ครั้ง

In [3]:
"""การค้นหาและการนับ — งานส่งวิชา Data Structures & Algorithms ของ Neo

อัลกอริทึมที่เขียนเองของไฟล์นี้คือ **binary search** ห้ามใช้ bisect ของ Python
ส่วน linear search เขียนไว้เพื่อเป็นตัวเทียบ และ hash lookup ใช้ตารางแฮช
ที่ Kim เขียนใน hash_table.py

ทั้งสามวิธีตอบคำถามเดียวกัน ("อุปกรณ์รหัสนี้คือชิ้นไหน") แต่ใช้เวลาต่างกัน
คนละระดับ ซึ่งเป็นสิ่งที่รายงานต้องพิสูจน์ด้วยเวลาที่วัดมาจริง ไม่ใช่แค่อ้าง Big-O

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

   ข้อนี้รวมถึง datasets.py และ hash_table.py ของคนในกลุ่มด้วย
   ไฟล์นี้จึงไม่ import อะไรเลย แต่ "รับ" ของที่ต้องใช้เข้ามาทางพารามิเตอร์
       - ชุดข้อมูล รับเป็น list ของ dict
       - ตารางแฮช รับเป็นออบเจ็กต์ที่มีเมท็อด .get() (ตัวของ Kim มีอยู่แล้ว)
   ใน Colab ให้วางเซลล์ตามลำดับ datasets -> hash_table -> ไฟล์นี้
   แล้วทุกอย่างจะอยู่ใน namespace เดียวกันเอง

วิธีรัน
    python3 backend-api/algorithms/search.py
    (รันลำพังได้ มีข้อมูลตัวอย่างเล็กๆ ของตัวเองอยู่ท้ายไฟล์)

ทำไมยังต้องมี binary search ทั้งที่ตารางแฮชเร็วกว่า
    ตารางแฮชเร็วเฉพาะตอนที่รู้ code เป๊ะๆ เพราะมันคำนวณตำแหน่งจาก key ตรงๆ
    แต่คำถามที่ร้านถามจริงคือ "มีของอะไรบ้างที่ครบกำหนดคืนก่อนวันที่ X"
    ตารางแฮชตอบไม่ได้เลย ต้องไล่ดูทั้งตาราง ซึ่งกลายเป็น O(n)
    คำถามแบบเป็นช่วงแบบนี้ต้องใช้ list ที่เรียงแล้ว + binary search
    ซึ่งหาขอบของช่วงได้ใน O(log n) — ดู find_due_before() ท้ายส่วนที่ 2
"""

import time

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — การค้นหา
# ---------------------------------------------------------------------------


def linear_search(items: list, code: str):
    """ไล่ดูทีละชิ้นตั้งแต่ต้นจนเจอ — O(n)

    กรณีดีที่สุดคือเจอที่ชิ้นแรก (1 ครั้ง) กรณีแย่ที่สุดคือไม่มีของชิ้นนั้นอยู่เลย
    ซึ่งต้องดูครบทุกชิ้นก่อนจะสรุปได้ว่าไม่มี (n ครั้ง)
    เฉลี่ยแล้วอยู่ประมาณกลาง list คือ n/2 ครั้ง ซึ่งยังโตตาม n จึงนับเป็น O(n)

    ข้อดีที่ทำให้ยังต้องมี: ใช้กับ list แบบไหนก็ได้ ไม่ต้องเรียงมาก่อน
    และไม่ต้องสร้างโครงสร้างอะไรเพิ่ม
    """
    for item in items:
        if item["code"] == code:
            return item

    return None


def binary_search(sorted_items: list, code: str):
    """ค้นบน list ที่เรียงตาม code แล้ว — O(log n) เขียนเอง ไม่ใช้ bisect

    วิธีคิด: ดูตัวที่อยู่ตรงกลางก่อน
        ถ้าตรงกลางคือตัวที่หา                -> จบ
        ถ้าตัวที่หา "น้อยกว่า" ตรงกลาง       -> ของที่หาต้องอยู่ครึ่งซ้าย ทิ้งครึ่งขวา
        ถ้าตัวที่หา "มากกว่า" ตรงกลาง        -> ของที่หาต้องอยู่ครึ่งขวา ทิ้งครึ่งซ้าย
    ทุกครั้งที่เทียบหนึ่งรอบ จำนวนที่เหลือต้องดูลดลงครึ่งหนึ่ง
    ข้อมูล 1,000 ชิ้นจึงใช้การเทียบแค่ประมาณ 10 ครั้ง (เพราะ 2^10 = 1,024)
    เทียบกับ linear search ที่ต้องเทียบเฉลี่ย 500 ครั้ง

    🔴 เงื่อนไขที่ห้ามลืม: list ต้องเรียงตาม code มาก่อนแล้ว
       ถ้าไม่เรียง การตัดครึ่งจะตัดผิดข้าง แล้วจะคืน None ทั้งที่ของอยู่ในนั้น
       ซึ่งเป็นบั๊กที่ไม่ฟ้องอะไรเลย ใช้ sort_by_code() ข้างล่างเตรียมก่อนเสมอ

    เขียนเป็นลูป ไม่ใช่เรียกตัวเอง (recursion) เพราะลูปไม่กินพื้นที่ stack เพิ่ม
    พื้นที่ที่ใช้จึงเป็น O(1) ไม่ใช่ O(log n)
    """
    low = 0
    high = len(sorted_items) - 1

    while low <= high:
        # เขียนแบบนี้แทน (low + high) // 2 เพื่อกันค่าล้นในภาษาที่ int มีขนาดจำกัด
        # ใน Python ไม่ล้นอยู่แล้ว แต่เขียนให้ถูกหลักไว้ เพราะเป็นท่ามาตรฐาน
        middle = low + (high - low) // 2
        current = sorted_items[middle]["code"]

        if current == code:
            return sorted_items[middle]

        if current < code:
            # ตัวกลางน้อยกว่าที่หา ของที่หาต้องอยู่ทางขวา
            low = middle + 1
        else:
            high = middle - 1

    return None


def hash_lookup(table, code: str):
    """ค้นผ่านตารางแฮชที่ Kim เขียน — O(1) โดยเฉลี่ย

    table คือออบเจ็กต์ HashTable จาก hash_table.py ส่งเข้ามาทางพารามิเตอร์
    ไม่ได้ import เพราะกฎ stdlib ที่หัวไฟล์ (ดูคำอธิบายด้านบน)

    ทำไมเป็น O(1) "เฉลี่ย" ไม่ใช่ทุกกรณี: ตารางแฮชคำนวณว่า key ควรอยู่ถังไหน
    แล้วไปหยิบจากถังนั้นเลย ถ้าถังนั้นมีของชิ้นเดียวก็จบใน 1 ครั้ง
    แต่ถ้า key หลายตัวคำนวณได้ถังเดียวกัน (ชนกัน) ถังนั้นจะยาวขึ้นและต้องไล่ในถัง
    กรณีแย่สุดคือทุก key ตกถังเดียวกันหมด ซึ่งจะกลายเป็น O(n) เท่า linear search
    """
    return table.get(code)


def search_by_name(items: list, keyword: str) -> list:
    """ค้นจากชื่อบางส่วน คืน "ทุก" ตัวที่ตรง — O(n)

    ต้องคืนเป็น list ไม่ใช่ตัวเดียว เพราะชื่อซ้ำกันได้จริงในร้าน
    (ชุดข้อมูล B มีไว้พิสูจน์ข้อนี้โดยเฉพาะ: ชื่อซ้ำเป็นคู่ แต่ code ไม่ซ้ำ)
    ถ้าคืนตัวแรกที่เจอเหมือน linear_search ผู้ใช้จะได้ของผิดชิ้นโดยไม่รู้ตัว
    ซึ่งแปลว่าปล่อยเช่าผิดชิ้น และของอีกชิ้นจะหายไปจากระบบเงียบๆ

    ค้นแบบนี้ใช้ตารางแฮชไม่ได้ เพราะแฮชต้องการ key ที่ตรงเป๊ะทั้งตัว
    "กล้อง" กับ "กล้อง รุ่นที่ 5" คำนวณได้คนละถังโดยสิ้นเชิง
    """
    if not keyword:
        return []

    needle = keyword.strip().lower()
    found = []

    for item in items:
        if needle in item["name"].lower():
            found.append(item)

    return found


def sort_by_code(items: list) -> list:
    """เรียงตาม code เพื่อเตรียมให้ binary_search

    ใช้ sorted() ของ Python ตรงนี้ได้ เพราะการเรียงไม่ใช่อัลกอริทึมที่ไฟล์นี้
    ต้องพิสูจน์ (เป็นงานของ Kul ใน sort_group.py) ตรงนี้แค่เตรียมข้อมูล
    ตอนวัดเวลา การเรียงจึงต้องทำ "ก่อน" เริ่มจับเวลาเสมอ ไม่งั้นจะกลายเป็น
    การวัดเวลาเรียง + ค้น แล้ว binary search จะดูแพ้ทั้งที่ไม่ได้ช้า

    คืน list ใหม่ ไม่แก้ของเดิม เพราะคนอื่นถือ list ใบเดียวกันอยู่
    """
    return sorted(items, key=lambda item: item["code"])


def find_due_before(sorted_by_due: list, limit_date: str) -> list:
    """หาทุกรายการที่ครบกำหนดคืน "ก่อน" วันที่กำหนด — O(log n + k)

    นี่คือคำถามที่ร้านถามจริงทุกเช้า และเป็นเหตุผลว่าทำไมยังต้องมี binary search
    ทั้งที่ตารางแฮชเร็วกว่า: แฮชตอบคำถามแบบ "เป็นช่วง" ไม่ได้เลย

    วิธี: ใช้การตัดครึ่งหาขอบซ้ายสุดที่ due_date >= limit_date ด้วย O(log n)
    แล้วตัดเอาทุกตัวก่อนหน้านั้น ซึ่งคือคำตอบทั้งก้อน
    k คือจำนวนคำตอบ ซึ่งต้องเสียเวลาก็อปออกมาอยู่แล้วไม่ว่าใช้วิธีไหน

    sorted_by_due ต้องเรียงตาม due_date แล้ว และต้องไม่มี None ปนอยู่
    (ของที่ไม่มีใครยืมไม่มีกำหนดคืน จึงไม่ควรอยู่ในรายการนี้ตั้งแต่ต้น)
    """
    low = 0
    high = len(sorted_by_due)

    # หาตำแหน่งแรกสุดที่ due_date >= limit_date
    # สังเกตว่า high เริ่มที่ len ไม่ใช่ len-1 เพราะคำตอบอาจเป็น "ท้ายสุด"
    # คือไม่มีตัวไหนถึงเกณฑ์เลย ทุกตัวจึงเข้าเงื่อนไขทั้งหมด
    while low < high:
        middle = low + (high - low) // 2

        if sorted_by_due[middle]["due_date"] < limit_date:
            low = middle + 1
        else:
            high = middle

    return sorted_by_due[:low]


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — การนับ
#
# ทุกตัวเป็น O(n) เพราะต้องดูทุกชิ้นถึงจะนับได้ ไม่มีทางลัด
# การนับต่างจากการค้นตรงที่ไม่มี "ตัวที่ใช่" ให้หยุด จึงข้ามตัวไหนไม่ได้เลย
# ---------------------------------------------------------------------------


def count_available(items: list) -> int:
    """นับจำนวน "ชิ้น" ที่ยังอยู่บนชั้นทั้งร้าน — O(n)

    นับเป็นชิ้น ไม่ใช่นับจำนวนรายการ เพราะหนึ่งรายการมีได้หลายชิ้น
    (กล้องรุ่นเดียวกัน 5 ตัว = 1 รายการ 5 ชิ้น)
    ตัวเลขนี้คือเลขที่ขึ้นบนแถบสถานะของหน้า workspace ในตัวเว็บ

    ของชำรุดไม่นับ ถึงจะยังวางอยู่บนชั้นก็ปล่อยเช่าไม่ได้
    ถ้านับรวม หน้าเว็บจะบอกว่ามีของว่าง 3 ชิ้นแต่กดปล่อยเช่าไม่ได้สักชิ้น
    """
    total = 0

    for item in items:
        if item["status"] != "broken":
            total += item["available"]

    return total


def count_by_status(items: list) -> dict:
    """นับจำนวน "รายการ" แยกตามสถานะ — O(n)

    คืนคีย์ครบทั้งสามเสมอ แม้ค่าจะเป็น 0 เพราะถ้าคีย์หายไปเฉยๆ
    คนที่เอาไปใช้ต่อต้องเขียน .get() ดักทุกที่ ซึ่งจะมีคนลืมสักที่แน่นอน

    เดินผ่านข้อมูลรอบเดียวได้ทั้งสามตัวเลข ไม่ใช่วนสามรอบแยกกัน
    ผลลัพธ์เท่ากันแต่วนรอบเดียวอ่านข้อมูลน้อยกว่าสามเท่า
    """
    counts = {"ok": 0, "out": 0, "broken": 0}

    for item in items:
        status = item["status"]
        if status in counts:
            counts[status] += 1

    return counts


def count_overdue(items: list, today: str) -> int:
    """นับรายการที่เลยกำหนดคืนแล้ว — O(n)

    today ส่งเข้ามาเป็นข้อความ "YYYY-MM-DD" ไม่ได้อ่านวันจริงจากเครื่อง
    เพราะตัวเลขในรายงานต้องเท่าเดิมทุกครั้งที่รัน ไม่ใช่เปลี่ยนไปทุกวัน
    (datasets.py ตรึงวันอ้างอิงไว้ที่ TODAY ให้แล้ว)

    เทียบวันด้วยการเทียบข้อความได้เลย เพราะรูปแบบ YYYY-MM-DD เรียงจาก
    หน่วยใหญ่ไปหน่วยเล็ก "2026-09-26" < "2026-10-01" เป็นจริงทั้งในเชิง
    ข้อความและเชิงวันที่ ถ้าเก็บเป็น "26/09/2026" ข้อนี้จะไม่จริงทันที

    ของที่คืนแล้วไม่มี due_date (เป็น None) จึงต้องข้ามก่อนเทียบ
    ไม่งั้นจะเทียบ None กับข้อความแล้ว Python จะโยน TypeError ทิ้ง
    """
    count = 0

    for item in items:
        due = item["due_date"]
        if due is not None and due < today:
            count += 1

    return count


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — เครื่องมือวัดเวลา
#
# ใช้วิธีเดียวกับ hash_table.py ของ Kim (perf_counter เฉลี่ย 3 รอบ)
# ตัวเลขของสองไฟล์จะได้อ่านเทียบกันได้ในรายงานฉบับเดียว
# ---------------------------------------------------------------------------

ROUNDS = 3  # รันซ้ำกี่รอบแล้วเฉลี่ย — กันจังหวะที่เครื่องไปทำอย่างอื่นชั่วคราว


def measure(operation, rounds: int = ROUNDS) -> float:
    """วัดเวลาเฉลี่ยของ operation หน่วยเป็นมิลลิวินาที

    ใช้ time.perf_counter() ไม่ใช่ time.time() เพราะ perf_counter เป็นนาฬิกา
    ความละเอียดสูงที่ทำมาเพื่อวัดช่วงเวลาสั้นๆ และไม่กระโดดเวลาเครื่องปรับนาฬิกา
    """
    durations = []

    for _ in range(rounds):
        started_at = time.perf_counter()
        operation()
        durations.append(time.perf_counter() - started_at)

    return sum(durations) / len(durations) * 1000


def lookup_order(items: list) -> list:
    """คืนลำดับ code ที่จะใช้ค้น — สลับแบบตายตัวโดยไม่ต้องสุ่ม

    ทำไมต้องสลับ: ถ้าไล่ค้นตามลำดับเดิม linear_search จะเจอที่ตำแหน่งต้นๆ
    เกือบทุกครั้ง ซึ่งเป็นกรณีดีที่สุดของมัน ตัวเลขจะสวยเกินจริงและสรุปผิด
    ว่า linear ไม่ได้ช้าอย่างที่ Big-O บอก

    ใช้การเดินข้ามทีละ 7 ช่อง (7 เป็นจำนวนเฉพาะ จึงวนครบทุกตำแหน่งพอดี
    ก่อนจะกลับมาซ้ำ) แทน random.shuffle เพื่อให้ได้ลำดับเดิมทุกครั้ง
    โดยไม่ต้อง import random เพิ่ม
    """
    size = len(items)
    return [items[(index * 7) % size]["code"] for index in range(size)]


def benchmark_search(items: list, table=None) -> dict:
    """วัดเวลา "ค้นทุก code หนึ่งครั้ง" ของแต่ละวิธี บนข้อมูลชุดเดียวกัน

    table ส่งเข้ามาได้ถ้ามี HashTable ของ Kim อยู่ใน namespace แล้ว
    ถ้าไม่ส่ง จะวัดแค่ linear กับ binary — ไฟล์นี้จึงรันลำพังได้

    การเรียงทำก่อนเริ่มจับเวลา เพราะในระบบจริง list ที่เรียงแล้วถูกเก็บไว้
    และใช้ซ้ำได้หลายพันครั้ง ไม่ได้เรียงใหม่ทุกครั้งที่มีคนค้น
    ถ้านับเวลาเรียงเข้าไปด้วย จะเป็นการวัดคนละอย่างกับที่ระบบทำจริง
    """
    codes = lookup_order(items)
    sorted_items = sort_by_code(items)

    def with_linear():
        for code in codes:
            linear_search(items, code)

    def with_binary():
        for code in codes:
            binary_search(sorted_items, code)

    results = {
        "linear": measure(with_linear),
        "binary": measure(with_binary),
    }

    if table is not None:

        def with_hash():
            for code in codes:
                hash_lookup(table, code)

        results["hash"] = measure(with_hash)

    return results


def print_table(title: str, rows: list) -> None:
    """พิมพ์ผลเป็นตารางอ่านง่าย — rows คือ list ของ tuple

    (ชื่อชุด, จำนวนข้อมูล, เวลา linear, เวลา binary, เวลา hash หรือ None)
    """
    print()
    print(title)
    print("-" * 78)
    print(
        f"{'ชุด':>5} {'จำนวน':>9} {'linear (ms)':>14} {'binary (ms)':>14}"
        f" {'hash (ms)':>12} {'เร็วขึ้น':>10}"
    )
    print("-" * 78)

    for name, size, linear_ms, binary_ms, hash_ms in rows:
        speedup = linear_ms / binary_ms if binary_ms > 0 else 0
        hash_text = f"{hash_ms:>12.3f}" if hash_ms is not None else f"{'-':>12}"
        print(
            f"{name:>5} {size:>9,} {linear_ms:>14.3f} {binary_ms:>14.3f}"
            f" {hash_text} {speedup:>9.1f}x"
        )

    print("-" * 78)


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — ข้อมูลตัวอย่างของไฟล์นี้เอง
#
# ชุดข้อมูล A-D ที่ใช้ส่งงานจริงอยู่ที่ datasets.py — เรียก make_dataset()
# ฟังก์ชันข้างล่างนี้มีไว้ให้ก็อปไฟล์เดียวไปวางใน Colab แล้วกดรันได้ทันที
# โดยไม่ต้องพาไฟล์อื่นไปด้วย ตัวเลขในรายงานต้องมาจาก datasets.py เท่านั้น
# ---------------------------------------------------------------------------

SAMPLE_TODAY = "2026-10-01"


def make_sample_items(count: int) -> list:
    """สร้างรายการจำลอง count ชิ้น — รูปแบบเดียวกับที่ datasets.py ผลิต

    ไม่สุ่มเลย เดินเป็นแบบแผนตายตัว จึงได้ข้อมูลเดิมทุกครั้งโดยไม่ต้องใส่ seed
    """
    categories = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]
    prefixes = {"กล้อง": "CAM", "เลนส์": "LEN", "ขาตั้ง": "TRI", "ไฟ": "LED", "ไมโครโฟน": "MIC"}
    items = []

    for number in range(1, count + 1):
        category = categories[number % len(categories)]
        total = (number % 5) + 1
        available = number % (total + 1)
        borrowed = total - available

        items.append(
            {
                "code": f"{prefixes[category]}-{number:04d}",
                "name": f"{category} รุ่นที่ {number}",
                "category": category,
                "total": total,
                "available": available,
                "borrower": f"ลูกค้า {number}" if borrowed > 0 else None,
                # สลับกันระหว่างเลยกำหนดกับยังไม่ถึงกำหนด
                "due_date": ("2026-09-20" if number % 3 == 0 else "2026-10-20") if borrowed > 0 else None,
                "status": "broken" if number % 7 == 0 else ("out" if available == 0 else "ok"),
            }
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — ตรวจว่าทุกฟังก์ชันทำงานถูก
# ---------------------------------------------------------------------------


def verify() -> None:
    """ตรวจผลของทุกฟังก์ชัน ถ้าไม่ตรงให้ล้มทันที

    จุดที่ตั้งใจตรวจเป็นพิเศษคือ binary_search ต้องให้คำตอบ "ตรงกับ"
    linear_search ทุกตัว ถ้าสองตัวนี้ตอบไม่ตรงกัน แปลว่าตัวใดตัวหนึ่งผิด
    และการเทียบเวลาในรายงานก็ไม่มีความหมาย เพราะเทียบของที่ทำงานคนละอย่าง
    """
    items = make_sample_items(40)
    sorted_items = sort_by_code(items)

    # binary ต้องตอบเหมือน linear ทุกตัว รวมถึงตัวที่ไม่มีอยู่จริงด้วย
    for item in items:
        assert binary_search(sorted_items, item["code"]) == linear_search(items, item["code"])

    for missing in ("ไม่มีจริง", "CAM-9999", "", "AAA-0001", "ZZZ-9999"):
        assert linear_search(items, missing) is None
        assert binary_search(sorted_items, missing) is None, f"binary หา {missing!r} แล้วต้องไม่เจอ"

    # ขอบของ list — ตัวแรกสุดและตัวท้ายสุดคือจุดที่ off-by-one ชอบโผล่
    assert binary_search(sorted_items, sorted_items[0]["code"]) == sorted_items[0]
    assert binary_search(sorted_items, sorted_items[-1]["code"]) == sorted_items[-1]

    # list ว่างและ list ที่มีตัวเดียว ต้องไม่ระเบิด
    assert binary_search([], "CAM-0001") is None
    single = [sorted_items[0]]
    assert binary_search(single, single[0]["code"]) == single[0]
    assert binary_search(single, "ZZZ-0000") is None

    # ค้นด้วยชื่อ ต้องได้ทุกตัวที่ตรง ไม่ใช่ตัวแรกที่เจอ
    duplicates = [
        {"code": "CAM-0001", "name": "กล้อง A", "category": "กล้อง", "total": 1,
         "available": 1, "borrower": None, "due_date": None, "status": "ok"},
        {"code": "CAM-0002", "name": "กล้อง A", "category": "กล้อง", "total": 1,
         "available": 0, "borrower": "ก", "due_date": "2026-09-01", "status": "out"},
    ]
    found = search_by_name(duplicates, "กล้อง A")
    assert len(found) == 2, "ชื่อซ้ำต้องคืนครบทุกตัว"
    assert {row["code"] for row in found} == {"CAM-0001", "CAM-0002"}
    assert search_by_name(duplicates, "") == []
    assert search_by_name(duplicates, "ไม่มีชื่อนี้") == []

    # นับของว่าง — ของชำรุดต้องไม่ถูกนับรวม
    hand_counted = sum(item["available"] for item in items if item["status"] != "broken")
    assert count_available(items) == hand_counted
    assert count_available([]) == 0

    # นับตามสถานะ — ผลรวมต้องเท่าจำนวนรายการทั้งหมด และมีครบสามคีย์เสมอ
    counts = count_by_status(items)
    assert set(counts) == {"ok", "out", "broken"}
    assert sum(counts.values()) == len(items)
    assert count_by_status([]) == {"ok": 0, "out": 0, "broken": 0}

    # นับของเลยกำหนด
    hand_overdue = sum(
        1 for item in items if item["due_date"] is not None and item["due_date"] < SAMPLE_TODAY
    )
    assert count_overdue(items, SAMPLE_TODAY) == hand_overdue
    assert count_overdue([], SAMPLE_TODAY) == 0

    # คำถามแบบเป็นช่วง — ตัวที่พิสูจน์ว่าทำไมยังต้องมี binary search
    with_due = sorted(
        [item for item in items if item["due_date"] is not None],
        key=lambda item: item["due_date"],
    )
    early = find_due_before(with_due, SAMPLE_TODAY)
    assert all(item["due_date"] < SAMPLE_TODAY for item in early)
    assert len(early) == hand_overdue, "หาแบบช่วงต้องได้จำนวนเท่ากับการนับทีละตัว"
    assert find_due_before(with_due, "2000-01-01") == []
    assert len(find_due_before(with_due, "2999-12-31")) == len(with_due)

    print("ตรวจฟังก์ชันค้นหาและนับ ผ่านทั้งหมด")


# ---------------------------------------------------------------------------
# ส่วนที่ 6 — รันเพื่อดูผล
# ---------------------------------------------------------------------------


def main() -> None:
    print("=" * 78)
    print("การค้นหาและการนับ — LEBIS (งานของ Neo)")
    print("=" * 78)

    verify()

    items = make_sample_items(20)
    print()
    print("ตัวอย่างผลการนับ บนข้อมูลตัวอย่าง 20 รายการ")
    print("-" * 78)
    print(f"    ของว่างบนชั้น        {count_available(items)} ชิ้น")
    print(f"    แยกตามสถานะ          {count_by_status(items)}")
    print(f"    เลยกำหนดคืน          {count_overdue(items, SAMPLE_TODAY)} รายการ")

    # วัดเวลาบนขนาดที่ต่างกัน ให้เห็นว่า linear แย่ลงเร็วแค่ไหนเมื่อข้อมูลโต
    # ตัวเลขจริงที่ใช้ในรายงานต้องมาจากชุด A-D ของ datasets.py
    rows = []
    for size in (10, 100, 1000):
        sample = make_sample_items(size)
        result = benchmark_search(sample)
        rows.append((str(size), size, result["linear"], result["binary"], None))

    print_table("เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) — ข้อมูลตัวอย่างของไฟล์นี้", rows)

    print()
    print("อ่านตารางนี้ยังไง")
    print("-" * 78)
    print("    ข้อมูลโตขึ้น 10 เท่า linear ใช้เวลาโตขึ้นประมาณ 100 เท่า")
    print("    เพราะทั้งจำนวนครั้งที่ค้นและความยาวที่ต้องไล่ โตขึ้นอย่างละ 10 เท่า")
    print("    ส่วน binary โตขึ้นแค่ประมาณ 10 เท่า เพราะแต่ละครั้งยังใช้การเทียบ")
    print("    เพิ่มขึ้นแค่ 3-4 ครั้งเท่านั้น (log2 ของ 10 เท่า)")
    print()
    print("    ตัวเลขที่ใช้ในรายงานจริงต้องมาจากชุด A-D ของ datasets.py")
    print("    วิธีรันใน Colab: วางเซลล์ datasets -> hash_table -> ไฟล์นี้")
    print("    แล้วเรียก benchmark_search(make_dataset('D'), build_hash_table(...))")


ตรวจว่าทุกฟังก์ชันทำงานถูก — จุดสำคัญคือ **binary ต้องตอบตรงกับ linear ทุกตัว**
ถ้าสองตัวนี้ตอบไม่ตรงกัน การเทียบเวลาก็ไม่มีความหมายเพราะเทียบของที่ทำงานคนละอย่าง

In [4]:
verify()   # verify() ของ search.py (ทับตัวของ datasets ไปแล้ว — ตั้งใจ)

ตรวจฟังก์ชันค้นหาและนับ ผ่านทั้งหมด


---
## 3. เวลาที่วัดได้จริงบนชุด A–D

วัด "ค้นครบทุก code หนึ่งรอบ" ของแต่ละวิธี เฉลี่ย 3 รอบ (`ROUNDS = 3`)
การเรียงทำ **ก่อน** เริ่มจับเวลา เพราะในระบบจริง list ที่เรียงแล้วถูกเก็บไว้ใช้ซ้ำ ไม่ได้เรียงใหม่ทุกครั้งที่ค้น

> ตัวเลข **เวลา** ขึ้นกับเครื่องและโหลด ณ ตอนรัน จึงต่างกันได้ในแต่ละครั้ง — สิ่งที่คงที่คือ **อัตราส่วน**
> และรูปทรงการโต (O(n) เทียบ O(log n)) ส่วนตัวเลข **นับ** (overdue/broken/…) คงที่เป๊ะทุกครั้งเพราะ `TODAY` ถูกตรึง

In [5]:
rows = []
for name in ("A", "B", "C", "D"):
    items = make_dataset(name)
    r = benchmark_search(items)          # linear + binary (ไม่ต้องมี hash table ของ Kim)
    rows.append((name, len(items), r["linear"], r["binary"], None))

print_table("เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) — ชุด A–D", rows)


เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) — ชุด A–D
------------------------------------------------------------------------------
  ชุด     จำนวน    linear (ms)    binary (ms)    hash (ms)   เร็วขึ้น
------------------------------------------------------------------------------
    A        10          0.003          0.004            -       0.7x
    B        12          0.004          0.005            -       0.7x
    C        10          0.003          0.004            -       0.7x
    D     1,000         14.913          1.197            -      12.5x
------------------------------------------------------------------------------


### อ่านตารางนี้ยังไง
- **ชุด A (10 รายการ):** linear เร็วกว่าหรือพอ ๆ กับ binary — ข้อมูลน้อยเกินกว่าที่การคำนวณตำแหน่งกลาง (`low + (high-low)//2`) จะคุ้ม
- **ชุด D (1,000 รายการ):** binary เร็วกว่า linear หลายเท่า (ดูคอลัมน์ `เร็วขึ้น`) — นี่คือ O(n) เทียบ O(log n) ที่วัดออกมาเป็นตัวเลขได้จริง
- ถ้าวัดแค่ชุด A ชุดเดียวจะ **สรุปผิดทั้งหัวข้อ** ว่า binary ไม่ได้เร็วกว่า

---
## 4. ผลการนับบนชุด A–D

ตัวเลขกลุ่มนี้ **คงที่ทุกครั้ง** เพราะไม่อ่านวันจริงจากเครื่อง ใช้ `TODAY = 2026-10-01` ที่ตรึงไว้

In [6]:
print(f"{'ชุด':>4} {'ว่างบนชั้น':>10} {'ok':>5} {'out':>5} {'broken':>7} {'เลยกำหนด':>10}")
print("-" * 50)
for name in ("A", "B", "C", "D"):
    items = make_dataset(name)
    by = count_by_status(items)
    print(f"{name:>4} {count_available(items):>10,} {by['ok']:>5} {by['out']:>5} "
          f"{by['broken']:>7} {count_overdue(items, TODAY_STR):>10}")

 ชุด ว่างบนชั้น    ok   out  broken   เลยกำหนด
--------------------------------------------------
   A         24     9     0       1          3
   B         14     7     4       1          4
   C         10     5     4       1          4
   D      1,309   616   242     142        275


---
## 5. คำถามแบบ "ช่วง" ที่ตารางแฮชตอบไม่ได้ — `find_due_before`

คำถามที่ร้านถามจริงทุกเช้า: *"มีของอะไรบ้างที่ครบกำหนดคืนก่อนวันที่ X"*
ตารางแฮชคำนวณตำแหน่งจาก key ที่ **ตรงเป๊ะ** เท่านั้น คำถามแบบช่วงจึงต้องไล่ทั้งตาราง กลายเป็น O(n)
ส่วน list ที่เรียงตาม `due_date` + binary search หาขอบของช่วงได้ใน **O(log n + k)** — นี่คือเหตุผลที่ยังต้องมี binary search

In [7]:
big = make_dataset("D")

# เตรียม: เอาเฉพาะตัวที่มีกำหนดคืน แล้วเรียงตาม due_date (ข้อความ YYYY-MM-DD เทียบตรง ๆ ได้)
with_due = sorted((it for it in big if it["due_date"] is not None), key=lambda it: it["due_date"])

# หาแบบช่วงด้วย binary search — O(log n + k)
early = find_due_before(with_due, TODAY_STR)

# ตรวจสอบด้วยการนับทีละตัวแบบ O(n) — ต้องได้เลขเดียวกัน
by_count = count_overdue(big, TODAY_STR)

print(f"ครบกำหนดคืนก่อน {TODAY_STR}")
print(f"    find_due_before (O(log n + k)) : {len(early)} รายการ")
print(f"    count_overdue   (O(n))          : {by_count} รายการ")
print(f"    ตรงกัน: {len(early) == by_count}")
print()
print("ตัวอย่าง 5 รายการแรกที่ครบกำหนด:")
for it in early[:5]:
    print(f"    {it['code']}  ครบกำหนด {it['due_date']}  ({it['name']})")

ครบกำหนดคืนก่อน 2026-10-01
    find_due_before (O(log n + k)) : 275 รายการ
    count_overdue   (O(n))          : 275 รายการ
    ตรงกัน: True

ตัวอย่าง 5 รายการแรกที่ครบกำหนด:
    MIC-0004  ครบกำหนด 2026-09-21  (ไมโครโฟน รุ่นที่ 4)
    LEN-0081  ครบกำหนด 2026-09-21  (เลนส์ รุ่นที่ 81)
    MIC-0104  ครบกำหนด 2026-09-21  (ไมโครโฟน รุ่นที่ 104)
    CAM-0125  ครบกำหนด 2026-09-21  (กล้อง รุ่นที่ 125)
    TRI-0142  ครบกำหนด 2026-09-21  (ขาตั้ง รุ่นที่ 142)


---
## 6. เทียบ hand-written กับของที่ Python มีให้ (ข้อกำหนดรายงาน ข้อ 4)

โจทย์ให้เทียบตัวที่เขียนเองกับ built-in ของ Python บนชุด D เพื่อยืนยันสองอย่าง:
(1) ที่เขียนมือ **ทำงานถูก** — ให้คำตอบเดียวกับ built-in ทุกตัว
(2) เข้าใจว่า **ทำไม** built-in เร็วกว่า (ไม่ใช่เพราะอัลกอริทึมต่างกัน แต่เพราะเขียนด้วย C ระดับล่าง)

- `binary_search` (เขียนเอง) เทียบ `bisect` ของ Python — **อัลกอริทึมเดียวกัน O(log n)** ต่างแค่ค่าคงที่
- `linear_search` (เขียนเอง, O(n)) เทียบการค้นผ่าน `dict` ของ Python (O(1)) — แทนสิ่งที่ตารางแฮชของ Kim ทำได้

> `bisect` และ `dict` ใช้ตรงนี้ได้ในฐานะ **ตัวเทียบเวลาเท่านั้น** ไม่ได้ใช้แทนใน implementation ที่ถูกให้คะแนน

In [8]:
import bisect, time

big = make_dataset("D")
codes = lookup_order(big)                       # ลำดับค้นแบบตายตัว (สลับแล้ว) จาก search.py
sorted_items = sort_by_code(big)
sorted_codes = [it["code"] for it in sorted_items]   # list ของ code ล้วน สำหรับ bisect
by_code = {it["code"]: it for it in big}        # dict ของ Python — ตัวแทน "ตารางแฮช"

# --- ตรวจความถูกต้องก่อนวัดเวลา: เขียนเองต้องได้คำตอบเดียวกับ built-in ---
def bisect_lookup(code):
    i = bisect.bisect_left(sorted_codes, code)
    if i < len(sorted_codes) and sorted_codes[i] == code:
        return sorted_items[i]
    return None

ok = all(binary_search(sorted_items, c) == bisect_lookup(c) == by_code.get(c) for c in codes)
print(f"เขียนเอง = bisect = dict ทุกตัว: {ok}\n")

# --- วัดเวลา เฉลี่ย 3 รอบ (ใช้ measure() ตัวเดียวกับที่ใช้ทั้งรายงาน) ---
t_binary = measure(lambda: [binary_search(sorted_items, c) for c in codes])
t_bisect = measure(lambda: [bisect_lookup(c) for c in codes])
t_linear = measure(lambda: [linear_search(big, c) for c in codes])
t_dict   = measure(lambda: [by_code.get(c) for c in codes])

print(f"{'วิธี':<34}{'เวลา (ms)':>12}")
print("-" * 46)
print(f"{'binary_search — เขียนเอง  O(log n)':<34}{t_binary:>12.3f}")
print(f"{'bisect — built-in         O(log n)':<34}{t_bisect:>12.3f}")
print(f"{'linear_search — เขียนเอง  O(n)':<34}{t_linear:>12.3f}")
print(f"{'dict.get — built-in       O(1)':<34}{t_dict:>12.3f}")

เขียนเอง = bisect = dict ทุกตัว: True

วิธี                                 เวลา (ms)
----------------------------------------------
binary_search — เขียนเอง  O(log n)       1.064
bisect — built-in         O(log n)       0.338
linear_search — เขียนเอง  O(n)          15.463
dict.get — built-in       O(1)           0.043


### สรุปผลข้อ 6
- **binary_search เขียนเอง vs `bisect`:** เวลาอยู่ระดับเดียวกัน (ทั้งคู่ O(log n)) `bisect` เร็วกว่าเล็กน้อยเพราะเขียนด้วย C — ไม่ใช่เพราะอัลกอริทึมดีกว่า พิสูจน์ว่าที่เขียนมือ **ถูกทั้งผลและระดับความเร็ว**
- **linear_search vs `dict.get`:** `dict` (O(1)) ทิ้งห่าง linear (O(n)) มาก บนชุด D — นี่คือเหตุผลที่โปรเจกต์มีตารางแฮชของ Kim ไว้สำหรับการค้นด้วย code ที่ตรงเป๊ะ
- แต่ `dict`/แฮช ตอบคำถามแบบ **ช่วง** (ข้อ 5) ไม่ได้ จึงยังต้องมี `binary_search` + list ที่เรียงแล้วอยู่ดี

---
## สรุป Big-O ทั้งโมดูล

| การทำงาน | วิธีที่เลือกใช้ | เวลา | เหตุผล |
|---|---|---|---|
| ค้นด้วย code ที่ตรงเป๊ะ | hash lookup (ของ Kim) | O(1) เฉลี่ย | เร็วสุดเมื่อรู้ key |
| ค้นด้วย code บน list ที่เรียงแล้ว | **binary_search (เขียนเอง)** | O(log n) | ไม่ต้องสร้างตารางแฮช |
| ค้นด้วยชื่อบางส่วน | `search_by_name` | O(n) | ชื่อซ้ำได้ ต้องคืนทุกตัว |
| ของครบกำหนดก่อนวันที่ X (ช่วง) | **`find_due_before` (binary)** | O(log n + k) | แฮชตอบแบบช่วงไม่ได้ |
| นับของว่าง/สถานะ/เลยกำหนด | `count_*` | O(n) | ต้องดูทุกชิ้น |

**ปิดท้าย:** ตัวเลขทุกตัวในโน้ตบุ๊กนี้มาจาก `make_dataset()` ชุด A–D ตัวเดียวกับที่ Kim และ Kul ใช้
รายงานสามหัวข้อจึงอ่านเทียบกันได้ในฉบับเดียว